# Week 11 — Ablation Study + Streamlit Demo

**Learning Goals:**
- Run systematic ablation experiments to isolate each contribution
- Build an interactive Streamlit demo for the final presentation

**Deliverables:**
- Ablation table: window size, SG smoothing, attention heads, MC samples
- Working Streamlit app under `app/`
- Final comparison plot (all models)

In [ ]:
import sys, torch, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_train, load_test
from preprocess import preprocess_pipeline
from models.cnn_transformer import CNNTransformerModel
from models.cnn_lstm import CNNLSTMModel
from models.lstm import LSTMModel
from train import train_model, compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Ablation: Window Size

In [ ]:
train_df = load_train('../data', 'FD001')
test_df, rul_true = load_test('../data', 'FD001')

window_sizes = [10, 20, 30, 40, 50]
ablation_window = []

for ws in window_sizes:
    print(f'\n--- Window size: {ws} ---')
    result = preprocess_pipeline(train_df, test_df, window_size=ws)
    X_tr, y_tr = result['X_train'], result['y_train']
    X_val, y_val = result['X_val'], result['y_val']
    X_te = result['X_test']
    
    model = CNNTransformerModel(
        n_features=14, seq_len=ws,
        cnn_channels=64, d_model=128, nhead=4, num_layers=2, dropout=0.2
    ).to(device)
    
    # Quick train (reduce epochs for ablation)
    history = train_model(
        model, X_tr, y_tr, X_val, y_val,
        epochs=30, batch_size=256, lr=1e-3, device=device,
        save_path=None, patience=10
    )
    
    # Evaluate on test
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_te, dtype=torch.float32).to(device)).cpu().numpy().flatten()
    
    metrics = compute_metrics(rul_true, preds)
    ablation_window.append({'window': ws, **metrics})
    print(f'  MAE: {metrics["mae"]:.2f}, RMSE: {metrics["rmse"]:.2f}, Score: {metrics["score"]:.0f}')

df_ws = pd.DataFrame(ablation_window)
print('\n', df_ws.to_string(index=False))

## 2. Ablation: SG Smoothing On/Off

In [ ]:
ablation_sg = []
for use_sg in [False, True]:
    print(f'\n--- SG Smoothing: {use_sg} ---')
    result = preprocess_pipeline(
        train_df, test_df, window_size=30,
        apply_sg=use_sg  # you may need to add this param
    )
    X_tr, y_tr = result['X_train'], result['y_train']
    X_val, y_val = result['X_val'], result['y_val']
    X_te = result['X_test']
    
    model = CNNTransformerModel(
        n_features=14, seq_len=30,
        cnn_channels=64, d_model=128, nhead=4, num_layers=2, dropout=0.2
    ).to(device)
    
    history = train_model(
        model, X_tr, y_tr, X_val, y_val,
        epochs=30, batch_size=256, lr=1e-3, device=device,
        save_path=None, patience=10
    )
    
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_te, dtype=torch.float32).to(device)).cpu().numpy().flatten()
    
    metrics = compute_metrics(rul_true, preds)
    ablation_sg.append({'sg_smoothing': use_sg, **metrics})

df_sg = pd.DataFrame(ablation_sg)
print('\n', df_sg.to_string(index=False))

## 3. Ablation: Number of Attention Heads

In [ ]:
result = preprocess_pipeline(train_df, test_df, window_size=30)
X_tr, y_tr = result['X_train'], result['y_train']
X_val, y_val = result['X_val'], result['y_val']
X_te = result['X_test']

head_configs = [1, 2, 4, 8]
ablation_heads = []

for nh in head_configs:
    print(f'\n--- Attention heads: {nh} ---')
    model = CNNTransformerModel(
        n_features=14, seq_len=30,
        cnn_channels=64, d_model=128, nhead=nh, num_layers=2, dropout=0.2
    ).to(device)
    
    history = train_model(
        model, X_tr, y_tr, X_val, y_val,
        epochs=30, batch_size=256, lr=1e-3, device=device,
        save_path=None, patience=10
    )
    
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_te, dtype=torch.float32).to(device)).cpu().numpy().flatten()
    
    metrics = compute_metrics(rul_true, preds)
    ablation_heads.append({'nhead': nh, **metrics})

df_heads = pd.DataFrame(ablation_heads)
print('\n', df_heads.to_string(index=False))

## 4. Final Model Comparison Table

In [ ]:
# Load results from all weeks (adjust paths as needed)
# This cell is a placeholder — fill in with actual saved metrics

comparison = pd.DataFrame([
    {'Model': 'Linear Regression',  'MAE': None, 'RMSE': None, 'Score': None, 'Week': 2},
    {'Model': 'Random Forest',      'MAE': None, 'RMSE': None, 'Score': None, 'Week': 2},
    {'Model': 'GBM',                'MAE': None, 'RMSE': None, 'Score': None, 'Week': 2},
    {'Model': 'LSTM',               'MAE': None, 'RMSE': None, 'Score': None, 'Week': 4},
    {'Model': 'CNN+LSTM (raw)',      'MAE': None, 'RMSE': None, 'Score': None, 'Week': 5},
    {'Model': 'CNN+LSTM (SG)',       'MAE': None, 'RMSE': None, 'Score': None, 'Week': 5},
    {'Model': 'CNN-Transformer',     'MAE': None, 'RMSE': None, 'Score': None, 'Week': 6},
    {'Model': 'CNN-Transformer+UQ',  'MAE': None, 'RMSE': None, 'Score': None, 'Week': 6},
])

print('Fill in the results from previous weeks!')
print(comparison.to_string(index=False))

## 5. Streamlit App

The Streamlit app lives in `app/streamlit_app.py`.

**To run:**
```bash
cd app
streamlit run streamlit_app.py
```

The app provides:
1. **Single Engine Analysis** — enter engine ID, get RUL prediction + explanation
2. **Fleet Dashboard** — overview of all engines, color-coded by risk
3. **Model Comparison** — side-by-side results from different architectures
4. **Uncertainty Inspector** — MC Dropout visualization

In [ ]:
# Quick test that the Streamlit app file is present
app_path = '../app/streamlit_app.py'
assert os.path.exists(app_path), f'Streamlit app not found at {app_path}'
print(f'✓ Streamlit app found: {app_path}')
print(f'  Run with: streamlit run {app_path}')